# 02 — Construction des variables à chaque date de prédiction

**Projet :** Cert_EET — prédiction précoce des apprenants à risque
**Auteur :** Iyed Omri — PFE ESPRIT 2026

## Objectif
Je veux prédire le risque **pendant** la formation, pas à la fin. Je choisis donc des **dates de
prédiction** (dates de coupure) : J+14, J+28, J+42 et J+56, en jours depuis le début de la session.

À chaque date, je calcule les variables de chaque apprenant **encore inscrit**, en utilisant
**seulement ce qui s'est passé avant cette date**. C'est la règle la plus importante du projet :
si le modèle voit des informations du futur, il paraît excellent pendant les tests mais il est
inutile en vrai. On appelle ce problème une **fuite de données**.

J'ai écrit le calcul dans un fichier Python séparé (`ml/risque/variables.py`) et non dans le notebook.
Ainsi, le notebook, les tests automatiques et l'entraînement utilisent **exactement le même code**.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Je rends le dossier ml/ importable, que le notebook tourne depuis ml/ ou ml/notebooks/
RACINE_ML = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "risque").exists())
sys.path.insert(0, str(RACINE_ML))

from risque.variables import (CLE, VARIABLES, charger_tables, preparer,
                              construire_variables, tronquer)

d = preparer(charger_tables(RACINE_ML / "donnees"))
print("Inscriptions :", len(d["inscriptions"]))
print("Jours actifs (étudiant × jour) :", len(d["jours"]))
print("Remises d'évaluations (hors examens) :", len(d["remises"]))

Inscriptions : 32593
Jours actifs (étudiant × jour) : 1808119
Remises d'évaluations (hors examens) : 167044


## 1. Les variables retenues

Chaque variable existe aussi dans Cert_EET (voir mon audit des données). Je n'utilise ni les clics,
ni le module, ni les variables personnelles.

| Variable | Sens | Règle Cert_EET proche |
|---|---|---|
| `jours_actifs` | nombre de jours avec au moins une activité avant J | — |
| `taux_jours_actifs` | jours actifs depuis le début ÷ jours écoulés | — |
| `jours_depuis_derniere_activite` | jours depuis la dernière activité (maximum : J) | `INACTIVITE` |
| `jours_actifs_14_derniers` | activité des deux dernières semaines | `INACTIVITE` |
| `delai_inscription` | jours entre l'inscription et le début (négatif = inscrit en avance) | — |
| `nb_evaluations_echues` | évaluations dont la date limite est passée | — |
| `taux_remise` | évaluations rendues ÷ évaluations échues | `LIVRABLE_EN_RETARD` |
| `nb_non_rendues` | évaluations échues et non rendues | `LIVRABLE_EN_RETARD` |
| `nb_remises_en_retard` | travaux rendus après la date limite | — |
| `nb_notes`, `score_moyen` | nombre de notes connues et leur moyenne (sur 100) | — |
| `nb_echecs`, `echecs_consecutifs_fin` | notes < 40, et série d'échecs la plus récente | `ECHECS_QCM` |
| `tendance_score` | dernière note − moyenne des précédentes | `SCORE_EN_CHUTE` |
| `a_au_moins_une_note` | 1 si l'apprenant a déjà une note | — |

**Deux choix importants :**
- quand il n'y a pas encore de note, je laisse `score_moyen` **vide** au lieu de mettre 0 : une
  absence de note n'est pas une mauvaise note. L'indicateur `a_au_moins_une_note` permet au modèle
  de faire la différence ;
- une évaluation dont la date limite n'est pas encore passée ne compte **ni** comme rendue, **ni**
  comme non rendue.

In [2]:
COUPURES = [14, 28, 42, 56]

tableaux = [construire_variables(d, J) for J in COUPURES]
variables = pd.concat(tableaux, ignore_index=True)

resume = (variables.groupby("coupure")
                   .agg(apprenants=("id_student", "count"), taux_risque=("a_risque", "mean"))
                   .assign(taux_risque=lambda t: (t["taux_risque"] * 100).round(1)))
print(resume)
print()
print("Valeurs manquantes :")
print(variables[VARIABLES].isna().sum()[lambda s: s > 0])

         apprenants  taux_risque
coupure                         
14            28069         45.3
28            27535         44.1
42            27022         43.1
56            26523         42.0

Valeurs manquantes :
score_moyen       41363
tendance_score    81984
dtype: int64


## 2. Preuve qu'il n'y a pas de fuite de données

Je ne veux pas seulement **affirmer** que mes variables n'utilisent pas le futur : je veux le
**prouver**.
Mon test : pour chaque date J, je crée une copie des données où **j'efface tout ce qui arrive à
partir du jour J** (activité, remises, désinscriptions, et même le résultat final). Je calcule les
variables sur cette copie, puis sur les données complètes.

**Si les deux tableaux sont identiques, mes variables ne dépendent d'aucune information future.**
Si une seule valeur diffère, le test échoue et me montre où se trouve la fuite.

In [3]:
for J in COUPURES:
    complet = construire_variables(d, J)[CLE + VARIABLES].sort_values(CLE).reset_index(drop=True)
    passe_seul = construire_variables(tronquer(d, J), J)[CLE + VARIABLES].sort_values(CLE).reset_index(drop=True)
    pd.testing.assert_frame_equal(complet, passe_seul)
    print(f"J+{J} : aucune fuite — {len(complet):,} apprenants, valeurs identiques")

J+14 : aucune fuite — 28,069 apprenants, valeurs identiques
J+28 : aucune fuite — 27,535 apprenants, valeurs identiques
J+42 : aucune fuite — 27,022 apprenants, valeurs identiques
J+56 : aucune fuite — 26,523 apprenants, valeurs identiques


## 3. Les variables séparent-elles déjà les deux groupes ?

Avant d'entraîner un modèle, je compare la moyenne de chaque variable entre les apprenants à risque
et les autres, à la date J+28.

In [4]:
j28 = variables[variables["coupure"] == 28]
comparaison = (j28.groupby("a_risque")[VARIABLES].mean().T
                  .rename(columns={0: "Non à risque", 1: "À risque"}).round(2))
print(comparaison)

# Sauvegarde pour l'étape suivante (dans donnees/, donc jamais versionné)
variables.to_csv(RACINE_ML / "donnees" / "variables.csv.gz", index=False)
print("\nFichier enregistré :", RACINE_ML / "donnees" / "variables.csv.gz")

a_risque                        Non à risque  À risque
jours_actifs                           16.84     11.23
taux_jours_actifs                       0.45      0.30
jours_depuis_derniere_activite          3.90      7.31
jours_actifs_14_derniers                6.44      4.29
delai_inscription                     -66.55    -66.39
nb_evaluations_echues                   0.83      0.90
taux_remise                             0.97      0.78
nb_non_rendues                          0.03      0.23
nb_remises_en_retard                    0.16      0.21
nb_notes                                0.93      0.72
score_moyen                            77.44     66.31
nb_echecs                               0.03      0.07
echecs_consecutifs_fin                  0.03      0.07
tendance_score                         -0.01     -1.84
a_au_moins_une_note                     0.78      0.64

Fichier enregistré : C:\Users\omriy\Desktop\cert_eet_repository\CERT_EET\ml\donnees\variables.csv.gz


## 4. Mes constats

**1. Moins d'apprenants, et un taux de risque plus bas, à chaque date.** À J+14, il reste 28 069
apprenants encore inscrits (contre 32 593 au total), puis 26 523 à J+56. Le taux de risque passe de
52,8 % (toutes inscriptions) à **45,3 % à J+14**, puis 42,0 % à J+56. C'est normal : j'ai retiré les
apprenants déjà partis, qui étaient tous « à risque ». Mon problème est donc **plus difficile, mais
plus honnête** : le modèle doit repérer ceux qui vont partir, pas ceux qui sont déjà partis.

**2. Aucune fuite de données.** Aux quatre dates, les variables calculées sur les données complètes
sont **strictement identiques** à celles calculées sur des données où tout le futur a été effacé.
C'est une preuve, pas seulement une affirmation.

**3. Les valeurs manquantes sont voulues.** `score_moyen` est vide quand l'apprenant n'a encore
aucune note, et `tendance_score` quand il a moins de deux notes. Ce n'est pas une erreur : c'est
mon choix de ne pas confondre « pas de note » et « mauvaise note ». Le modèle traitera ces cas à
l'étape suivante.

**4. À J+28, les variables séparent déjà bien les deux groupes :**
- **activité** : 16,8 jours actifs contre 11,2 ; dernière activité il y a 3,9 jours contre 7,3 ;
- **remises** : 97 % des travaux échus rendus contre 78 % ; 0,03 travail non rendu contre 0,23 ;
- **notes** : 77,4 de moyenne contre 66,3.

**5. Deux variables semblent peu utiles.** `delai_inscription` est presque identique dans les deux
groupes (−66,6 contre −66,4 jours) : la date d'inscription ne semble pas prédire le risque. Je la garde
pour l'instant ; l'importance des variables (étape SHAP) confirmera ou non ce constat.
`nb_evaluations_echues` dépend surtout du calendrier du cours, pas de l'apprenant : elle sert de
**contexte** pour interpréter `taux_remise`.

**6. Tôt dans la session, l'activité compte plus que les notes.** À J+28, un apprenant a en moyenne
moins d'une note. Pour les premières dates, le modèle s'appuiera donc surtout sur l'activité. Les
notes prendront plus de poids plus tard.